<a href="https://colab.research.google.com/github/lilmiztee04/instagram-content-efficiency-model/blob/main/TheImpactmodelContentefficiency.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>The impact model<h1>
<h3>Welcome to the impact model. In this notebook, I develop a content efficiency model designed to identify the most effective content themes for increasing Instagram post engagement and improving overall performance. <h3>


<h3>1. In this section, I will carry out data cleaning and make any required adjustments to ensure the dataset is accurate, consistent, and ready for analysis<h3>

In [1]:
#upload file, rename and check attributes
import pandas as pd
SME = pd.read_csv('/content/social_media_engagement1.csv')
display(SME.head())

,post_id,platform,post_type,post_time,likes,comments,shares,post_day,sentiment_score
0,1,Facebook,image,8/17/2023 14:45,2121,474,628,Thursday,positive
1,2,Facebook,carousel,5/14/2023 0:45,3660,432,694,Sunday,neutral
2,3,Instagram,poll,2/21/2023 16:15,4955,408,688,Tuesday,negative
3,4,Twitter,image,11/16/2023 0:45,1183,90,187,Thursday,negative
4,5,Twitter,video,5/23/2023 0:30,3499,247,286,Tuesday,positive


In [2]:
#check how many records are in the dataset
SME.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   post_id          100 non-null    int64 
 1   platform         100 non-null    object
 2   post_type        100 non-null    object
 3   post_time        100 non-null    object
 4   likes            100 non-null    int64 
 5   comments         100 non-null    int64 
 6   shares           100 non-null    int64 
 7   post_day         100 non-null    object
 8   sentiment_score  100 non-null    object
dtypes: int64(4), object(5)
memory usage: 7.2+ KB


In [3]:
#remove unnecessary attributes, twitter isnt needed as im focusing on meta platforms such as instagram and facebook
SME = SME[SME['platform'] != 'Twitter']
display(SME.head())

,post_id,platform,post_type,post_time,likes,comments,shares,post_day,sentiment_score
0,1,Facebook,image,8/17/2023 14:45,2121,474,628,Thursday,positive
1,2,Facebook,carousel,5/14/2023 0:45,3660,432,694,Sunday,neutral
2,3,Instagram,poll,2/21/2023 16:15,4955,408,688,Tuesday,negative
5,6,Instagram,carousel,5/5/2023 20:00,256,186,211,Friday,neutral
6,7,Instagram,image,2/26/2023 11:45,1982,30,906,Sunday,positive


In [4]:
#These attributes were dropped because they are not required for the engagement efficiency model and will be analysed separately.
SME = SME.drop(columns=['post_day','post_id', 'post_time'])
display(SME.head())

,platform,post_type,likes,comments,shares,sentiment_score
0,Facebook,image,2121,474,628,positive
1,Facebook,carousel,3660,432,694,neutral
2,Instagram,poll,4955,408,688,negative
5,Instagram,carousel,256,186,211,neutral
6,Instagram,image,1982,30,906,positive


In [5]:
#one hot encoding used to turn catagorical values into numerical values
sentiment_map = {'positive': 1, 'neutral': 0, 'negative': -1}
SME['sentiment_score'] = SME['sentiment_score'].map(sentiment_map)

In [6]:
#Feature engineering was required due to missing content theme data,
# which is a key variable for this project, theres 5 types of content themes needed, they where randomly assigned
import random

# Define a list of content themes based on user's input
content_themes = [
    'Informational/Educational',
    'Promotional (Product Specific)',
    'UGC (User Generated Content)',
    'Lifestyle/Brand Storytelling',
    'Trend-based'
]

# Randomly assign a content theme to each row in the SME DataFrame
SME['content_theme'] = [random.choice(content_themes) for _ in range(len(SME))]

display(SME.head())

,platform,post_type,likes,comments,shares,sentiment_score,content_theme
0,Facebook,image,2121,474,628,1,Trend-based
1,Facebook,carousel,3660,432,694,0,Lifestyle/Brand Storytelling
2,Instagram,poll,4955,408,688,-1,Informational/Educational
5,Instagram,carousel,256,186,211,0,Lifestyle/Brand Storytelling
6,Instagram,image,1982,30,906,1,Lifestyle/Brand Storytelling


In [7]:
#saved the dataset so it can be used for the main code
SME.to_csv('SMEcleaned.csv', index=False)
print('SME DataFrame saved to SMEcleaned.csv')

SME DataFrame saved to SMEcleaned.csv


<h3>2. Calculating engagement , sentiment, and weighted scores <h3>

In [8]:
#creating engagement score calculations
SMEP = pd.read_csv('SMEcleaned.csv')
SMEP['engagement'] = SMEP['likes'] + SMEP['comments'] + SMEP['shares']

In [9]:
#weighted calculations
#All metrics are weighted according to the project objectives and aims, with shares and comments given higher importance.
SMEP['weighted_engagement'] = (
    SMEP['likes'] +
    (2 * SMEP['comments']) +
    (3 * SMEP['shares'])
)

In [10]:
#import libary for scaler
from sklearn.preprocessing import MinMaxScaler
#MinMaxScaler to normalise engagement values between 0 and 1,
#this ensures consistency in scale and allows fair comparison across different features when analysing engagement rates.
scaler = MinMaxScaler()
SMEP['engagement_rate'] = scaler.fit_transform(
    SMEP[['weighted_engagement']]
)

In [11]:
#sentiment calculations
# Create adjusted engagement by factoring in sentiment, now we can see if the engagement is positive or negative or neutral
SMEP['sentiment_adjusted_engagement'] = (
    SMEP['engagement_rate'] * SMEP['sentiment_score']
)

In [12]:
SMEP.head()

,platform,post_type,likes,comments,shares,sentiment_score,content_theme,engagement,weighted_engagement,engagement_rate,sentiment_adjusted_engagement
0,Facebook,image,2121,474,628,1,Trend-based,3223,4953,0.515247,0.515247
1,Facebook,carousel,3660,432,694,0,Lifestyle/Brand Storytelling,4786,6606,0.723538,0.000000
2,Instagram,poll,4955,408,688,-1,Informational/Educational,6051,7835,0.878402,-0.878402
3,Instagram,carousel,256,186,211,0,Lifestyle/Brand Storytelling,653,1261,0.050025,0.000000
4,Instagram,image,1982,30,906,1,Lifestyle/Brand Storytelling,2918,4760,0.490927,0.490927


In [13]:
SMEP.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   platform                       68 non-null     object 
 1   post_type                      68 non-null     object 
 2   likes                          68 non-null     int64  
 3   comments                       68 non-null     int64  
 4   shares                         68 non-null     int64  
 5   sentiment_score                68 non-null     int64  
 6   content_theme                  68 non-null     object 
 7   engagement                     68 non-null     int64  
 8   weighted_engagement            68 non-null     int64  
 9   engagement_rate                68 non-null     float64
 10  sentiment_adjusted_engagement  68 non-null     float64
dtypes: float64(2), int64(6), object(3)
memory usage: 6.0+ KB


In [14]:
#These attributes were dropped because they are not required for the engagement efficiency model and will be analysed separately.
SMEP = SMEP.drop(columns=['engagement','sentiment_score','weighted_engagement','engagement_rate','likes','comments','shares'])
display(SME.head())

,platform,post_type,likes,comments,shares,sentiment_score,content_theme
0,Facebook,image,2121,474,628,1,Trend-based
1,Facebook,carousel,3660,432,694,0,Lifestyle/Brand Storytelling
2,Instagram,poll,4955,408,688,-1,Informational/Educational
5,Instagram,carousel,256,186,211,0,Lifestyle/Brand Storytelling
6,Instagram,image,1982,30,906,1,Lifestyle/Brand Storytelling


In [15]:
SMEP.head()

,platform,post_type,content_theme,sentiment_adjusted_engagement
0,Facebook,image,Trend-based,0.515247
1,Facebook,carousel,Lifestyle/Brand Storytelling,0.000000
2,Instagram,poll,Informational/Educational,-0.878402
3,Instagram,carousel,Lifestyle/Brand Storytelling,0.000000
4,Instagram,image,Lifestyle/Brand Storytelling,0.490927


In [16]:
SMEP.to_csv('SMEcleaned.csv', index=False)

<h3>3. Model<h3>

In [17]:
#libaries are used to run statistical analysis on the dataset ive selected
import numpy as np #used for mathematical calculations
import statsmodels.api as sm #used for statitical models
import statsmodels.formula.api as smf#used for logitic regresion model
from statsmodels.formula.api import ols #used for the anova model

#Building Anova model
#Modelling sentiment_adjusted_engagement based on platform , post type and content theme
anova_model = ols(
    'sentiment_adjusted_engagement ~ C(platform) + C(post_type) + C(content_theme)', data=SMEP).fit()
#runs anova tests based on modelling
anova_table = sm.stats.anova_lm(anova_model, typ=2)

#Displaying anova results table
print("=== ANOVA RESULTS ===")
print(anova_table)


#Interpretation of model(findings)
print(" ANOVA INTERPRETATION ")

#list to store the results found in the Anova table, translated and stored in thi variable significant_vars
significant_vars = []

#Anova interpretation loop, Goes through each factor in the model
for var in ['C(platform)', 'C(post_type)', 'C(content_theme)']:
  #Takes alll the p values in the table from each variable
    p_value = anova_table.loc[var, 'PR(>F)']
    #implement IFELSE statement
    #this is the significant test if p<0.05 it has a significant effect else its not significant
    #then prints message depending on outcome

    if p_value < 0.05:
        print(f"{var} is statistically significant (p = {p_value:.3f}) → affects engagement.")
        significant_vars.append(var)
    else:
        print(f"{var} is NOT significant (p = {p_value:.3f}) → no strong evidence of effect.")
#This means that none of the features impace engagement
if len(significant_vars) == 0:
    print("\nOverall: No content attributes significantly explain engagement differences.")
else:
    print(f"\nOverall: Significant factors include: {', '.join(significant_vars)}")


#Content theme analysis

#section header
print("\n=== CONTENT THEME ANALYSIS ===")
#groups dataset by content theme, calculates average engagement for each content theme then sorts from highest to lowest
theme_means = SMEP.groupby('content_theme')['sentiment_adjusted_engagement'].mean().sort_values(ascending=False)
#prints the means of the content
print(theme_means)
#Use this to find the best performing content theme and the worst performing content theme
best_theme = theme_means.idxmax()
worst_theme = theme_means.idxmin()
#printing the results calculated
print(f"\nTop performing theme: {best_theme}")
print(f"Lowest performing theme: {worst_theme}")


#!Logistic regresion model!

# Creates a new column called high enagement
#compares each post engagement  to the median engagement and if its above the median its true and below is false
SMEP['high_engagement'] = (
    SMEP['sentiment_adjusted_engagement'] > SMEP['sentiment_adjusted_engagement'].median()).astype(int)
#calculating binary values as LR cannot work with continuous values

#tests whether content attributes increase the likelihood of high engagement.
log_model = smf.logit(
    #is high engagement explained by platform, content type and content theme
    'high_engagement ~ C(platform) + C(post_type) + C(content_theme)',data=SMEP).fit()#training the model

print("\n=== LOGISTIC REGRESSION RESULTS ===")
print(log_model.summary())


#Logistic regression results

print("\n=== LOGISTIC INTERPRETATION ===")
#converts model results into a format that shows how each variable changes the likelihood of high engagemen
odds_ratios = np.exp(log_model.params)
#This tells us if the values are significant
p_values = log_model.pvalues
#creates a storage list
significant_log_vars = []
#creating a loop to go through every variable
for var in odds_ratios.index:

    if var == 'Intercept':
        continue
#if else statement of standard significant rule
    if p_values[var] < 0.05:
      #this variable determines if it increases enagement likelyhood or decreases
        direction = "increases" if odds_ratios[var] > 1 else "decreases"
        #Tells users the results from calculations
        print(f"{var} significantly {direction} likelihood of high engagement (OR = {odds_ratios[var]:.2f})")
        #saves them into a list
        significant_log_vars.append(var)
#if nothing was significant then it will post this,model found no strong predictors of engagement likelihood.
if len(significant_log_vars) == 0:
    print("No variables significantly influence the probability of high engagement.")


#Output of the results which anwer the research question

print("\n=== FINAL ANSWERS TO RESEARCH QUESTIONS ===")

# Primary question
if len(significant_vars) == 0 and len(significant_log_vars) == 0:
    print("Primary Question:")
    print("No content attributes were found to be strongly associated with higher engagement.")
else:
    print("Primary Question:")
    print("Some content attributes show significant relationships with engagement.")
    print(f"Key factors: {significant_vars + significant_log_vars}")

# Secondary question
print("\nSecondary Question:")
print(f"Content themes vary in engagement, with '{best_theme}' performing best and '{worst_theme}' performing worst.")

=== ANOVA RESULTS ===
                     sum_sq    df         F    PR(>F)
C(platform)        0.003088   1.0  0.010979  0.916913
C(post_type)       0.077005   4.0  0.068442  0.991202
C(content_theme)   0.816345   4.0  0.725566  0.578069
Residual          16.314166  58.0       NaN       NaN
 ANOVA INTERPRETATION 
C(platform) is NOT significant (p = 0.917) → no strong evidence of effect.
C(post_type) is NOT significant (p = 0.991) → no strong evidence of effect.
C(content_theme) is NOT significant (p = 0.578) → no strong evidence of effect.

Overall: No content attributes significantly explain engagement differences.

=== CONTENT THEME ANALYSIS ===
content_theme
Lifestyle/Brand Storytelling      0.127640
UGC (User Generated Content)      0.046715
Trend-based                      -0.021807
Informational/Educational        -0.066501
Promotional (Product Specific)   -0.310610
Name: sentiment_adjusted_engagement, dtype: float64

Top performing theme: Lifestyle/Brand Storytelling
Lowest perf